# Retail Data Cleaning

This notebook cleans and prepares the Online Retail dataset for business
analysis.

The original dataset will remain unchanged. Separate datasets will be created
for cleaned transactions, completed sales, and customer-level analysis.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

raw_data_path = (
    project_root/ "data"/ "raw"/ "Online Retail.xlsx"
)

processed_data_dir = (
    project_root/ "data"/ "processed"
)

reports_table_dir = (
    project_root/ "reports"/ "tables"
)

processed_data_dir.mkdir(
    parents=True,
    exist_ok=True,
)

reports_table_dir.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"Project root: {project_root}")
print(f"Dataset path: {raw_data_path}")
print(f"Dataset exists: {raw_data_path.exists()}")

Project root: c:\Users\ASUS\Python_workspace\retail_sales_insights
Dataset path: c:\Users\ASUS\Python_workspace\retail_sales_insights\data\raw\Online Retail.xlsx
Dataset exists: True


## 1. Load the Raw Dataset

In [3]:
raw_df = pd.read_excel(raw_data_path)

print("Dataset loaded successfully.")
print(f"Rows: {len(raw_df):,}")
print(f"Columns: {raw_df.shape[1]}")

Dataset loaded successfully.
Rows: 541,909
Columns: 8


In [4]:
raw_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 2. Create a Copy

The raw DataFrame will not be modified directly. All cleaning operations will
be applied to a separate copy.

In [5]:
clean_df = raw_df.copy()

print(f"Raw shape: {raw_df.shape}")
print(f"Working-copy shape: {clean_df.shape}")

Raw shape: (541909, 8)
Working-copy shape: (541909, 8)


## 3. Cleaning Rules

The following rules will be used:

1. Keep the original raw dataset unchanged.
2. Rename columns using snake_case.
3. Convert invoice dates into datetime values.
4. Convert quantity and unit price into numeric values.
5. Clean whitespace from text and identifier columns.
6. Remove rows missing essential transaction information.
7. Keep rows with missing customer IDs for general sales analysis.
8. Remove exact duplicate rows.
9. Mark invoice numbers beginning with 'C' as cancelled.
10. Keep cancellations and returns in the cleaned transaction dataset.
11. Create a separate dataset for valid completed sales.
12. Create another dataset for sales with known customer IDs.
13. Calculate revenue and useful date features.

## 4. Standardize Column Names

In [6]:
column_mapping = {
    "InvoiceNo": "invoice_no",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "UnitPrice": "unit_price",
    "CustomerID": "customer_id",
    "Country": "country",
}

clean_df = clean_df.rename(
    columns=column_mapping
)

clean_df.columns.tolist()

['invoice_no',
 'stock_code',
 'description',
 'quantity',
 'invoice_date',
 'unit_price',
 'customer_id',
 'country']

## 5. Correct Data Types

In [7]:
clean_df["invoice_date"] = pd.to_datetime(clean_df["invoice_date"],errors="coerce",)

In [8]:
clean_df["quantity"] = pd.to_numeric(clean_df["quantity"],errors="coerce",)

In [9]:
clean_df["unit_price"] = pd.to_numeric(clean_df["unit_price"],errors="coerce",)

In [10]:
clean_df["customer_id"] = (clean_df["customer_id"].astype("Int64"))

In [11]:
clean_df.dtypes

invoice_no              object
stock_code              object
description             object
quantity                 int64
invoice_date    datetime64[us]
unit_price             float64
customer_id              Int64
country                    str
dtype: object

## 6. Clean Text Columns

In [12]:
text_columns = ["invoice_no","stock_code","description","country",]

for column in text_columns:
    clean_df[column] = (clean_df[column].astype("string").str.strip())

In [13]:
clean_df[["invoice_no","stock_code","description","country",]].head()

,invoice_no,stock_code,description,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,United Kingdom
1,536365,71053,WHITE METAL LANTERN,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,United Kingdom


In [14]:
conversion_report = pd.Series({
    "missing_invoice_no": (
        clean_df["invoice_no"].isna().sum()
    ),
    "missing_stock_code": (
        clean_df["stock_code"].isna().sum()
    ),
    "missing_description": (
        clean_df["description"].isna().sum()
    ),
    "invalid_quantity": (
        clean_df["quantity"].isna().sum()
    ),
    "invalid_invoice_date": (
        clean_df["invoice_date"].isna().sum()
    ),
    "invalid_unit_price": (
        clean_df["unit_price"].isna().sum()
    ),
    "missing_customer_id": (
        clean_df["customer_id"].isna().sum()
    ),
    "missing_country": (
        clean_df["country"].isna().sum()
    ),
})

conversion_report

missing_invoice_no           0
missing_stock_code           0
missing_description       1454
invalid_quantity             0
invalid_invoice_date         0
invalid_unit_price           0
missing_customer_id     135080
missing_country              0
dtype: int64

## 7. Handle Missing Essential Values

Customer IDs are not considered essential for general sales analysis, so rows
with missing customer IDs will be retained.

Rows missing essential transaction fields will be removed.

In [15]:
essential_columns = ["invoice_no","stock_code","description","quantity","invoice_date","unit_price","country",]

In [16]:
missing_essential_mask = (clean_df[essential_columns].isna().any(axis=1))

missing_essential_count = (missing_essential_mask.sum())

print("Rows missing essential fields: "f"{missing_essential_count:,}")

Rows missing essential fields: 1,454


In [17]:
clean_df[missing_essential_mask].head(10)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
622,536414,22139,<NA>,56,2010-12-01 11:52:00,0.0,<NA>,United Kingdom
1970,536545,21134,<NA>,1,2010-12-01 14:32:00,0.0,<NA>,United Kingdom
1971,536546,22145,<NA>,1,2010-12-01 14:33:00,0.0,<NA>,United Kingdom
1972,536547,37509,<NA>,1,2010-12-01 14:33:00,0.0,<NA>,United Kingdom
1987,536549,85226A,<NA>,1,2010-12-01 14:34:00,0.0,<NA>,United Kingdom
1988,536550,85044,<NA>,1,2010-12-01 14:34:00,0.0,<NA>,United Kingdom
2024,536552,20950,<NA>,1,2010-12-01 14:34:00,0.0,<NA>,United Kingdom
2025,536553,37461,<NA>,3,2010-12-01 14:35:00,0.0,<NA>,United Kingdom
2026,536554,84670,<NA>,23,2010-12-01 14:35:00,0.0,<NA>,United Kingdom
2406,536589,21777,<NA>,-10,2010-12-01 16:50:00,0.0,<NA>,United Kingdom


In [18]:
rows_before_missing_removal = len(clean_df)

clean_df = clean_df.dropna(subset=essential_columns).copy()

rows_after_missing_removal = len(clean_df)

print("Rows removed: "f"{rows_before_missing_removal - rows_after_missing_removal:,}")

Rows removed: 1,454


In [19]:
empty_description_mask = (clean_df["description"].eq(""))

print("Empty descriptions: "f"{empty_description_mask.sum():,}")

Empty descriptions: 0


In [20]:
clean_df = clean_df[~empty_description_mask].copy()

## 8. Remove Exact Duplicate Rows

Only rows that are identical across every column will be treated as exact
duplicates.

In [21]:
duplicate_count = (clean_df.duplicated().sum())

print(f"Exact duplicate rows: {duplicate_count:,}")

Exact duplicate rows: 5,268


In [22]:
duplicate_rows = clean_df[clean_df.duplicated(keep=False)]

duplicate_rows.head(20)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
548,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920,United Kingdom


In [23]:
rows_before_duplicate_removal = len(clean_df)

clean_df = (clean_df.drop_duplicates().copy())

rows_after_duplicate_removal = len(clean_df)

duplicates_removed = (rows_before_duplicate_removal- rows_after_duplicate_removal)

print(f"Duplicates removed: {duplicates_removed:,}")

Duplicates removed: 5,268


## 9. Identify Cancellations and Returns

In [24]:
clean_df["is_cancelled"] = (clean_df["invoice_no"].str.startswith("C", na=False))

In [25]:
clean_df["is_cancelled"].value_counts()

is_cancelled
False    525936
True       9251
Name: count, dtype: Int64

In [26]:
clean_df.loc[clean_df["is_cancelled"],["invoice_no","description","quantity","unit_price",],].head(20)

,invoice_no,description,quantity,unit_price
141,C536379,Discount,-1,27.50
154,C536383,SET OF 3 COLOURED FLYING DUCKS,-1,4.65
235,C536391,PLASTERS IN TIN CIRCUS PARADE,-12,1.65
236,C536391,PACK OF 12 PINK PAISLEY TISSUES,-24,0.29
237,C536391,PACK OF 12 BLUE PAISLEY TISSUES,-24,0.29
238,C536391,PACK OF 12 RED RETROSPOT TISSUES,-24,0.29
239,C536391,CHICK GREY HOT WATER BOTTLE,-12,3.45
240,C536391,PLASTERS IN TIN VINTAGE PAISLEY,-12,1.65
241,C536391,PLASTERS IN TIN SKULLS,-24,1.65
939,C536506,JAM MAKING SET WITH JARS,-6,4.25


In [27]:
conditions = [clean_df["is_cancelled"],clean_df["quantity"] < 0,clean_df["unit_price"] <= 0,]

categories = ["cancelled","return","invalid_price",]

clean_df["transaction_type"] = np.select(conditions,categories,default="sale",)

In [28]:
clean_df["transaction_type"].value_counts()

transaction_type
sale             524878
cancelled          9251
invalid_price       584
return              474
Name: count, dtype: int64

## 10. Create Business Features

In [29]:
clean_df["line_revenue"] = (clean_df["quantity"]* clean_df["unit_price"])

In [30]:
clean_df[["quantity","unit_price","line_revenue","transaction_type",]].head(10)

,quantity,unit_price,line_revenue,transaction_type
0,6,2.55,15.30,sale
1,6,3.39,20.34,sale
2,8,2.75,22.00,sale
3,6,3.39,20.34,sale
4,6,3.39,20.34,sale
5,2,7.65,15.30,sale
6,6,4.25,25.50,sale
7,6,1.85,11.10,sale
8,6,1.85,11.10,sale
9,32,1.69,54.08,sale


In [31]:
clean_df["invoice_month"] = (clean_df["invoice_date"].dt.to_period("M").dt.to_timestamp())

In [32]:
clean_df["invoice_year"] = (clean_df["invoice_date"].dt.year)

clean_df["invoice_quarter"] = (clean_df["invoice_date"].dt.quarter)

clean_df["invoice_weekday"] = (clean_df["invoice_date"].dt.day_name())

clean_df["invoice_hour"] = (clean_df["invoice_date"].dt.hour)

clean_df["invoice_day"] = (clean_df["invoice_date"].dt.date)

clean_df["invoice_month"] = (clean_df["invoice_date"].dt.month_name())

In [33]:
clean_df[["invoice_date","invoice_month","invoice_year","invoice_quarter","invoice_weekday","invoice_hour",]].head()

,invoice_date,invoice_month,invoice_year,invoice_quarter,invoice_weekday,invoice_hour
0,2010-12-01 08:26:00,December,2010,4,Wednesday,8
1,2010-12-01 08:26:00,December,2010,4,Wednesday,8
2,2010-12-01 08:26:00,December,2010,4,Wednesday,8
3,2010-12-01 08:26:00,December,2010,4,Wednesday,8
4,2010-12-01 08:26:00,December,2010,4,Wednesday,8


## 11. Create the Valid Sales Dataset

Valid completed sales must have:

- A positive quantity
- A positive unit price
- An invoice that is not cancelled

In [34]:
valid_sales_mask = ((clean_df["quantity"] > 0)& (clean_df["unit_price"] > 0)& (~clean_df["is_cancelled"]))

In [35]:
sales_df = clean_df[valid_sales_mask].copy()

Check the size

In [36]:
print(f"Clean transaction rows: {len(clean_df):,}")

print(f"Valid sales rows: {len(sales_df):,}")

Clean transaction rows: 535,187
Valid sales rows: 524,878


Check transaction types

In [37]:
sales_df["transaction_type"].value_counts()

transaction_type
sale    524878
Name: count, dtype: int64

## 12. Create the Customer Sales Dataset

Rows with missing customer IDs will be excluded only from customer-level
analysis.

In [38]:
customer_sales_df = (sales_df.dropna(subset=["customer_id"]).copy())

In [39]:
print( "Valid sales rows: "f"{len(sales_df):,}")

print("Customer sales rows: "f"{len(customer_sales_df):,}")

print("Rows excluded from customer analysis: "f"{len(sales_df) - len(customer_sales_df):,}")

Valid sales rows: 524,878
Customer sales rows: 392,692
Rows excluded from customer analysis: 132,186


## 13. Cleaning Summary

In [40]:
cleaning_summary = pd.DataFrame({
    "dataset": [
        "Raw transactions",
        "After removing missing essential fields",
        "After removing exact duplicates",
        "Valid completed sales",
        "Customer sales",
    ],
    "row_count": [
        len(raw_df),
        rows_after_missing_removal,
        rows_after_duplicate_removal,
        len(sales_df),
        len(customer_sales_df),
    ],
})

In [41]:
cleaning_summary["percentage_of_raw"] = (cleaning_summary["row_count"]/ len(raw_df)* 100).round(2)

cleaning_summary

,dataset,row_count,percentage_of_raw
0,Raw transactions,541909,100.00
1,After removing missing essential fields,540455,99.73
2,After removing exact duplicates,535187,98.76
3,Valid completed sales,524878,96.86
4,Customer sales,392692,72.46


## 14. Validate the Cleaned Data

In [42]:
expected_columns = {
    "invoice_no",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "unit_price",
    "customer_id",
    "country",
    "is_cancelled",
    "transaction_type",
    "line_revenue",
    "invoice_month",
    "invoice_year",
    "invoice_quarter",
    "invoice_weekday",
    "invoice_hour",
    "invoice_day",
}

assert expected_columns.issubset(clean_df.columns)

assert clean_df.columns.is_unique

assert clean_df.duplicated().sum() == 0

assert clean_df[essential_columns].notna().all().all()

assert (sales_df["quantity"] > 0).all()

assert (sales_df["unit_price"] > 0).all()

assert (~sales_df["is_cancelled"]).all()

assert (sales_df["line_revenue"] > 0).all()

assert (customer_sales_df["customer_id"].notna().all())

print("All validation checks passed.")

All validation checks passed.


## 15. Export Processed Data

In [43]:
clean_transactions_path = (processed_data_dir/ "clean_transactions.csv")

valid_sales_path = (processed_data_dir/ "valid_sales.csv")

customer_sales_path = (processed_data_dir/ "customer_sales.csv")

cleaning_summary_path = (reports_table_dir/ "cleaning_summary.csv")

In [44]:
clean_df.to_csv(clean_transactions_path,index=False,)

sales_df.to_csv(valid_sales_path,index=False,)

customer_sales_df.to_csv(customer_sales_path,index=False,)

cleaning_summary.to_csv(cleaning_summary_path,index=False,)

print("Files saved successfully.")

print(clean_transactions_path)
print(valid_sales_path)
print(customer_sales_path)
print(cleaning_summary_path)

Files saved successfully.
c:\Users\ASUS\Python_workspace\retail_sales_insights\data\processed\clean_transactions.csv
c:\Users\ASUS\Python_workspace\retail_sales_insights\data\processed\valid_sales.csv
c:\Users\ASUS\Python_workspace\retail_sales_insights\data\processed\customer_sales.csv
c:\Users\ASUS\Python_workspace\retail_sales_insights\reports\tables\cleaning_summary.csv


In [45]:
print(clean_transactions_path.exists())
print(valid_sales_path.exists())
print(customer_sales_path.exists())
print(cleaning_summary_path.exists())

True
True
True
True


## 16. Conclusion

The raw dataset was cleaned without modifying the original Excel file.

Column names and data types were standardized, text fields were cleaned, and
exact duplicate rows were removed. Cancelled transactions, returns, and
records with invalid prices were identified instead of being mixed with valid
completed sales.

Rows with missing customer IDs were kept for general sales analysis because
they still contain useful product, country, and revenue information. They were
excluded only from the customer-specific dataset.

Revenue and date-related features were added for the next stage of the
project.